# Fase 5 — Regresión polinómica y regularización

**TP1 — Aprendizaje Automático (72.75) — ITBA** · Consignas **3.1, 3.2, 3.3** y **4**

Aplicamos la transformación polinómica a las variables de entrada, entrenamos la regresión
sobre las variables transformadas y evaluamos la regularización L1 con varios valores de
lambda. Toda la evaluación usa el mismo k-fold de la Fase 4, **sólo sobre el train**.

In [1]:
# ---------------------------------------------------------------------------
# Configuracion del entorno
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures

from src.config import TARGET, RANDOM_SEED
from src.data import cargar_splits
from src.preprocesamiento import separar_X_y, construir_preprocesador

# Funciones propias, documentadas en src/modelado.py
from src.modelado import (
    crear_kfold,                # el mismo KFold de la Fase 4
    crear_modelo_polinomico,    # Pipeline: preproc + PolynomialFeatures + escalado + regresion
    evaluar_cv,                 # validacion cruzada: RMSE de train y de validacion
    contar_features,            # cuantas columnas usa el modelo y cuantas quedan en cero
)

pd.set_option("display.width", 130)

train, _ = cargar_splits()
X_train, y_train = separar_X_y(train)
kf = crear_kfold()

print(f"Train: {X_train.shape[0]} filas | k-fold con {kf.get_n_splits()} folds")

Train: 1069 filas | k-fold con 5 folds


---

# 1. Transformación polinómica *(consigna 3.1)*

**Qué hace.** `PolynomialFeatures` toma las columnas de entrada y agrega **todas las
potencias hasta el grado elegido y todos los productos cruzados**. Con dos variables `a` y
`b` y grado 2, pasa de `[a, b]` a `[a, b, a², ab, b²]`.

**Por qué la necesitamos acá.** En la Fase 2 encontramos que el efecto de `bmi` sobre el
costo depende de si la persona fuma (correlación 0.086 entre no fumadores contra 0.809 entre
fumadores). Un modelo lineal suma efectos independientes y no puede representar eso: tiene un
único coeficiente para `bmi`, igual para todo el mundo.

El término cruzado `bmi × smoker_yes` que genera esta transformación es exactamente lo que
hace falta para que el modelo pueda tener dos comportamientos distintos.

In [2]:
# Cuantas columnas genera cada grado a partir de las 8 del preprocesamiento.
X_base = construir_preprocesador().fit_transform(X_train)

filas = []
for grado in (1, 2, 3, 4):
    n_cols = PolynomialFeatures(degree=grado, include_bias=False).fit_transform(X_base[:5]).shape[1]
    filas.append({
        "grado": grado,
        "n_features": n_cols,
        "filas_por_parametro": round(len(X_train) / n_cols, 1),
    })

pd.DataFrame(filas).set_index("grado")

,n_features,filas_por_parametro
grado,,
1,8,133.6
2,44,24.3
3,164,6.5
4,494,2.2


**Qué grados evaluamos: 1, 2 y 3.**

La consigna pide los grados *"que consideren oportuno"*. El criterio es la cantidad de datos
disponibles por parámetro: la Clase 2 (slide 91) plantea la "regla del 10×", unas 10
observaciones por parámetro.

| Grado | Features | Filas por parámetro | Decisión |
|---|---|---|---|
| 1 | 8 | 133.6 | Se evalúa (es el modelo de la Fase 4, sirve de referencia) |
| 2 | 44 | 24.3 | Se evalúa: holgado |
| 3 | 164 | 6.5 | Se evalúa: por debajo de 10, pero es justo donde la regularización puede ayudar |
| 4 | 494 | 2.2 | **Descartado sin probar**: menos de 3 observaciones por parámetro |

---

# 2. Entrenamiento sobre las variables transformadas *(consigna 3.2)*

Entrenamos la regresión lineal sobre las features polinómicas, sin regularizar todavía.

**Un detalle del pipeline:** después de la expansión polinómica volvemos a estandarizar. Las
columnas nuevas (cuadrados y productos) tienen escalas que no se parecen a las originales —
el cuadrado de una variable estandarizada ya no tiene media 0 ni desvío 1—, y sin reescalar
la penalización L1 de la sección siguiente castigaría a esas columnas por su magnitud en vez
de por su utilidad.

In [3]:
# Grados 1, 2 y 3 sin regularizacion.
resultados_sin_reg = []
for grado in (1, 2, 3):
    modelo = crear_modelo_polinomico(grado)
    _, resumen = evaluar_cv(modelo, X_train, y_train, f"Grado {grado}", cv=kf)
    resultados_sin_reg.append(resumen)

pd.DataFrame(resultados_sin_reg).set_index("modelo").round(2)

,rmse_train,rmse_val,rmse_val_desvio,gap,r2_val
modelo,,,,,
Grado 1,6075.92,6123.65,211.65,47.73,0.72
Grado 2,4748.62,4931.59,143.40,182.98,0.82
Grado 3,4514.11,5169.48,173.48,655.37,0.80


**La curva de complejidad, en tres puntos:**

| Grado | RMSE train | RMSE validación | Gap |
|---|---|---|---|
| 1 | 6.075,9 | 6.123,7 | **47,7** |
| 2 | 4.748,6 | **4.931,6** | 183,0 |
| 3 | 4.514,1 | 5.169,5 | **655,4** |

Es el comportamiento que describe la Clase 2 (slide 53): *underfitting → buen ajuste →
overfitting*.

- **Grado 1** tiene el gap más chico pero el peor error: el modelo es demasiado simple para el
  problema. Es **underfitting**.
- **Grado 2** baja el RMSE de validación un **19.5%** respecto del lineal (6.123,7 → 4.931,6).
  El gap sube a 183, todavía moderado.
- **Grado 3** sigue bajando el error de train (4.514,1, el más bajo de los tres) pero **el de
  validación empeora** (5.169,5) y el gap se dispara a 655. Está empezando a memorizar ruido:
  es **overfitting**.

El grado 3 es justamente el caso donde la regularización tiene algo que hacer.

---

# 3. Regularización L1 *(consigna 3.3)*

**Qué hace Lasso.** Modifica la función de costo agregando una penalización por el tamaño de
los coeficientes (Clase 3, slide 90):

$$\text{costo} = \text{error}(w) + \lambda \sum_j |w_j|$$

El modelo ya no minimiza sólo el error: tiene que equilibrar ajuste y simplicidad. Cuanto
mayor es lambda, más se lo fuerza a usar coeficientes chicos.

La penalización L1 tiene una propiedad particular: **lleva coeficientes exactamente a cero**.
Una variable con coeficiente cero no se usa, así que Lasso funciona además como método de
selección de variables (Clase 3, slide 90 y slide 63, familia *embedded*).

> **Sobre el nombre del parámetro.** En las clases se llama **lambda**; en scikit-learn se
> llama **`alpha`**. Es exactamente lo mismo: la fuerza de la penalización. En este notebook
> la columna se llama `lambda` para seguir la nomenclatura de la materia.

**Los valores evaluados.** La Clase 3 (slide 95) usa λ = 0.001, 0.01, 0.1 y 1. Los incluimos y
además agregamos 10, 100 y 1000, porque `charges` está en miles de dólares y los coeficientes
del modelo son del orden de los miles: valores de lambda menores que 1 pueden resultar
demasiado chicos para tener algún efecto.

---

# 4. Evaluación *(consigna 4)*

Evaluamos **todas las combinaciones** de grado y lambda con el mismo k-fold, y reportamos el
RMSE de train y de validación de cada una.

La celda siguiente es la búsqueda completa de hiperparámetros: **ningún modelo evaluado queda
fuera de la tabla**, incluidos los que pierden.

> Tarda unos 4 minutos. Los valores de lambda muy chicos son los más lentos: con una
> penalización casi nula el optimizador de Lasso necesita muchas iteraciones para converger.

In [4]:
# Busqueda completa: 3 grados sin regularizar + 2 grados x 7 valores de lambda.
LAMBDAS = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

resultados = []
for grado in (1, 2, 3):
    # None = sin regularizar; el resto son los valores de lambda a evaluar.
    valores = [None] if grado == 1 else [None] + LAMBDAS
    for lam in valores:
        modelo = crear_modelo_polinomico(grado, alpha=lam)

        # Error de train y de validacion promediados sobre los folds.
        _, resumen = evaluar_cv(modelo, X_train, y_train, "", cv=kf)

        # Entrenamos con todo el train solo para contar cuantas features
        # sobreviven a la penalizacion L1.
        modelo.fit(X_train, y_train)
        n_total, n_no_nulos = contar_features(modelo, X_train)

        resultados.append({
            "grado": grado,
            "lambda": lam,
            "rmse_train": round(resumen["rmse_train"], 1),
            "rmse_val": round(resumen["rmse_val"], 1),
            "gap": round(resumen["gap"], 1),
            "r2_val": round(resumen["r2_val"], 3),
            "features_usadas": f"{n_no_nulos}/{n_total}",
        })

tabla = pd.DataFrame(resultados)
tabla

,grado,lambda,rmse_train,rmse_val,gap,r2_val,features_usadas
0,1,NaN,6075.9,6123.7,47.7,0.723,8/8
1,2,NaN,4748.6,4931.6,183.0,0.820,41/44
2,2,0.001,4748.6,4931.6,183.0,0.820,41/44
3,2,0.010,4748.6,4931.6,183.0,0.820,41/44
4,2,0.100,4748.6,4931.5,182.9,0.820,41/44
5,2,1.000,4748.6,4930.8,182.2,0.820,41/44
6,2,10.000,4750.6,4924.5,173.9,0.820,41/44
7,2,100.000,4797.3,4905.3,108.1,0.822,20/44
8,2,1000.000,5271.7,5296.6,24.9,0.793,4/44
9,3,NaN,4514.1,5169.5,655.4,0.802,148/164


In [5]:
# El mejor modelo es el de menor RMSE de validacion.
mejor = tabla.loc[tabla["rmse_val"].idxmin()]

print("MODELO CON MENOR ERROR DE VALIDACION")
print(f"  Grado del polinomio : {int(mejor['grado'])}")
print(f"  Lambda (L1)         : {mejor['lambda']}")
print(f"  RMSE train          : {mejor['rmse_train']:,.1f}")
print(f"  RMSE validacion     : {mejor['rmse_val']:,.1f}")
print(f"  Gap                 : {mejor['gap']:,.1f}")
print(f"  R2 validacion       : {mejor['r2_val']}")
print(f"  Features usadas     : {mejor['features_usadas']}")

lineal = tabla.loc[(tabla['grado'] == 1)].iloc[0]
print(f"\nMejora sobre la regresion lineal de la Fase 4: "
      f"{100 * (lineal['rmse_val'] - mejor['rmse_val']) / lineal['rmse_val']:.1f}%")

MODELO CON MENOR ERROR DE VALIDACION
  Grado del polinomio : 2
  Lambda (L1)         : 100.0
  RMSE train          : 4,797.3
  RMSE validacion     : 4,905.3
  Gap                 : 108.1
  R2 validacion       : 0.822
  Features usadas     : 20/44

Mejora sobre la regresion lineal de la Fase 4: 19.9%


## 4.1 Qué muestra la tabla

**El ganador es grado 2 con λ = 100: RMSE de validación 4.905,3.**

**Los lambdas chicos no hacen nada.** De λ = 0.001 a λ = 1, el RMSE de validación del grado 2
va de 4.931,6 a 4.930,8: una diferencia de menos de un dólar. Con coeficientes del orden de
los miles, una penalización de 0.001 es despreciable. **Los valores que usa el ejemplo de la
Clase 3 son demasiado chicos para este dataset**, y el efecto sólo aparece a partir de λ = 10.

**La regularización rescata al grado 3.** Sin regularizar, el grado 3 daba un RMSE de
validación de 5.169,5 con un gap de 655. Con λ = 100 baja a **4.918,0** y el gap cae a 215.
Es la demostración de para qué sirve la regularización: le permite a un modelo con mucha
capacidad no memorizar el ruido.

**Con λ = 1000 se pasa de rosca.** Ambos grados saltan a ~5.296 y el gap cae a ~25. La
penalización es tan fuerte que casi todos los coeficientes van a cero (4 de 44 y 6 de 164) y
el modelo vuelve a ser demasiado simple: **underfitting otra vez**, ahora por exceso de
regularización.

**Lasso selecciona variables.** En el mejor modelo quedan **20 features de 44**: la
penalización descartó más de la mitad. En el grado 3 con λ = 100 quedan 49 de 164.

> Nota sobre la columna `features_usadas` en las filas sin regularizar: ahí los ceros no son
> selección sino consecuencia de columnas linealmente dependientes (el cuadrado de una
> variable binaria es igual a la variable), que el ajuste por mínimos cuadrados resuelve
> dejando algunos coeficientes en cero.

---

## Conclusiones de la Fase 5

| Consigna | Resultado |
|---|---|
| **3.1** Transformación polinómica | Grados 1, 2 y 3 evaluados. Grado 4 descartado por tener sólo 2.2 filas por parámetro |
| **3.2** Entrenamiento sobre las transformadas | Grado 2 mejora el RMSE de validación un 19.5% sobre el lineal |
| **3.3** Regularización L1 | 7 valores de lambda evaluados por grado, de 0.001 a 1000 |
| **4** Evaluación | Tabla completa con RMSE de train y validación por grado y lambda |

**Mejor modelo: grado 2 con λ = 100.**

| | Grado 1 (Fase 4) | Grado 2, λ = 100 |
|---|---|---|
| RMSE train | 6.075,9 | 4.797,3 |
| **RMSE validación** | **6.123,7** | **4.905,3** |
| Gap | 47,7 | 108,1 |
| R² validación | 0.723 | 0.822 |
| Features usadas | 8 de 8 | 20 de 44 |

El modelo polinómico reduce el error de validación un **19.9%** respecto del lineal, con un
gap todavía chico.

**Siguiente:** Fase 6 — evaluación final en test y comparación de modelos (consigna 5).